# Chicago Crime

**Author:** Erik Pak  
**Date:** 04/2026  
**Data:** Chicago Data Portal - Crime Incidents 2001–2025

___
## Scope:
We are modeling crime incidence as a multivariate non-stationary stochastic process and detecting crime type shifts via structural break detection across latent distributional parameters.

## Era Definitions

| Era        | Period                  | Months |
|------------|-------------------------|--------|
| Pre-COVID  | Jan 2001 – Feb 2020     | 230    |
| COVID      | Mar 2020 – Dec 2022     | 34     |
| Post-COVID | Jan 2023 – Dec 2025     | 36     |

Era cutoff rationale: The COVID era begins in March 2020, coinciding with the Illinois stay-at-home order (March 21, 2020) and the WHO pandemic declaration (March 11, 2020). The post-COVID era begins in January 2023, following the expiration of Illinois's disaster proclamation and the effective end of major federal pandemic-era policies in late 2022. These boundaries are administrative and policy-based; the underlying behavioral and enforcement shifts may not align exactly with these dates.

---
## Purpose: Detecting structural shocks directly from the data itself.
    * COVID-type disruptions
    * Policy changes
    * Economic changes
    * Policing changes
    * Social unrest
---
## Sudden changes in statistical behaviors:
    * Mean increases suddenly
    * Variance suddenly increases
    * Trend changes direction
    * Seasonality changes
    * Volatility shifts

In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

# Path + custom modules───
sys.path.append('../Src/')
import detection_config as cfg
import detection_analyze_fill as daf

# Library versions
versions = {
    "Python"  : sys.version.split()[0],
    "Pandas"  : pd.__version__,
    "NumPy"   : np.__version__,
    "Pyarrow" : pa.__version__,
    "Seaborn" : sns.__version__,
    "Matplot" : sys.modules['matplotlib'].__version__,
    "Scipy"   : sys.modules['scipy'].__version__,
}
df_versions = pd.DataFrame(list(versions.items()), columns=['Library', 'Version'])
print(df_versions)

   Library Version
0   Python  3.13.9
1   Pandas   2.3.3
2    NumPy   2.3.4
3  Pyarrow  22.0.0
4  Seaborn  0.13.2
5  Matplot  3.10.7
6    Scipy  1.16.3


## Data Import

In [2]:
# PyArrow's version of the 'arrow' backend
df = feather.read_feather('../Data/crime_data_covid.feather', memory_map=True, types_mapper=pd.ArrowDtype)
# display
df.head()

,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,beat,...,day_of_week,quarter,year_quarter,time_of_day,fbi_code_desc,fbi_index_code,district_location,year_week,year_month,Indexed
0,01G050460,2001-01-24 20:45:00,072XX S RIDGELAND AV,1811,NARCOTICS,POSS: CANNABIS 30GMS OR LESS,SIDEWALK,True,False,0324,...,Wednesday,Q1,2001-Q1,Night,Drug Abuse Violations,False,Grand Crossing,2001-04,200101,N
1,03J493690,2003-07-12 17:00:00,075XX S DOBSON AVE,0890,THEFT,FROM BUILDING,APARTMENT,False,False,0624,...,Saturday,Q3,2003-Q3,Evening,Larceny – Theft,True,Gresham,2003-28,200307,I
2,04X245238,2004-12-13 21:15:00,006XX N RIDGEWAY AVE,2024,NARCOTICS,POSS: HEROIN(WHITE),SIDEWALK,True,False,1122,...,Monday,Q4,2004-Q4,Night,Drug Abuse Violations,False,Harrison,2004-51,200412,N
3,07C115980,2006-03-31 09:15:00,026XX N NARRAGANSETT AVE,0610,BURGLARY,FORCIBLE ENTRY,APARTMENT,False,False,2512,...,Friday,Q1,2006-Q1,Morning,Burglary,True,Grand Central,2006-13,200603,I
4,07HN36467,2007-05-25 14:51:00,022XX N LA CROSSE AVE,1812,NARCOTICS,POSS: CANNABIS MORE THAN 30GMS,RESIDENCE,True,False,2522,...,Friday,Q2,2007-Q2,Afternoon,Drug Abuse Violations,False,Grand Central,2007-21,200705,N


In [3]:
df.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469443 entries, 0 to 8469442
Data columns (total 34 columns):
 #   Column                Non-Null Count    Dtype                                                       
---  ------                --------------    -----                                                       
 0   case_number           8469443 non-null  string[pyarrow]                                             
 1   date                  8469443 non-null  timestamp[s][pyarrow]                                       
 2   block                 8469443 non-null  string[pyarrow]                                             
 3   iucr                  8469443 non-null  string[pyarrow]                                             
 4   primary_type          8469443 non-null  string[pyarrow]                                             
 5   description           8469443 non-null  string[pyarrow]                                             
 6   location_description  8454105 non-

## Data Integration & Cleaning

In [4]:
# get the dates of period in the dataset
start_date = pd.to_datetime(df[cfg.config["_DATE_KEY"]].min(), format='%Y%m').date().strftime('%Y-%m')
end_date = pd.to_datetime(df[cfg.config["_DATE_KEY"]].max(), format='%Y%m').date().strftime('%Y-%m') 
# print the date range
print(f"Dataset covers from {start_date} to {end_date}")

Dataset covers from 2001-01 to 2025-12


### Check missingness & Fill missing data & Validate filled data

In [5]:
# Check integrity and missingness
integrity_results = daf.run_integrity_report(df)

# Fill missing data and report
fill_results = daf.fill_missing(df)

# Validate filled data
daf.validate_crime_data(fill_results['filled_df'])


📊 Crime Data Integrity Summary
----------------------------------------
Date range:       2001-01 to 2025-12
Total groups:     26
Expected rows:    7,800
Actual rows:      7,525
Missing rows:     275
Duplicates:       0
Completeness:     96.47%

🔍 True Missing Crime Gaps:
fbi_code_desc
Involuntary Manslaughter / Reckless Homicide    230
Gambling                                         25
Embezzlement                                     15
Prostitution                                      3
Stolen Property (Buy, Receive, Possess)           2

📉 Sparse / Risky Crime Groups:
fbi_code_desc
Involuntary Manslaughter / Reckless Homicide    0.233333
Gambling                                        0.916667
Embezzlement                                    0.950000
Prostitution                                    0.990000
Stolen Property (Buy, Receive, Possess)         0.993333

📊 Crime Data Filling Summary
Date Range:          2001-01 to 2025-12
Total Groups:        26
Total Periods:       300
To

In [6]:
# Info of the filled DataFrame
fill_results['filled_df'].info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7800 entries, 0 to 7799
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   fbi_code_desc       7800 non-null   category      
 1   year_month          7800 non-null   datetime64[ns]
 2   crime_count         7800 non-null   float64       
 3   was_missing         7800 non-null   bool          
 4   is_zero_after_fill  7800 non-null   bool          
dtypes: bool(2), category(1), datetime64[ns](1), float64(1)
memory usage: 146.1 KB


## Compositional Data Analysis (CoDA)

In [7]:
integrity_results.keys()

dict_keys(['date_range', 'missing', 'missing_by_group', 'coverage_ratio', 'duplicates', 'expected_rows', 'actual_rows', 'missing_rows', 'completeness'])

In [8]:
import pandas as pd

_DATE_KEY    = cfg.config["_DATE_KEY"]
_GROUP_KEY   = cfg.config["_GROUP_KEY"]
_COUNTER_KEY =cfg.config["_COUNTER_KEY"]


def _pivot(data_df, index, column, values):
    """
    Pivots long-format data into a wide matrix with optional caching.
    
    Args:
        data_df (pd.DataFrame): The source dataframe.
        index (str): Column to use as the new index (e.g., Date).
        column (str): Column to use as the new column (e.g., Crime Type).
        values (str): Column to populate the cells (e.g., Counts).
        
    Returns:
        pd.DataFrame: The T x K pivot table.
    """
    # 1. Execution
    # Note: Using .pivot_table() instead of .pivot() is safer if there's 
    # any chance of duplicate (index, column) pairs; it aggregates by default.
    pivot_df = (
        data_df
        .pivot(index=index, columns=column, values=values)
        .sort_index()
        .fillna(0) # Standard practice for count matrices to avoid NaN issues
    )

    return pivot_df

In [9]:
# Usage:
pivot = _pivot(fill_results['filled_df'],_DATE_KEY, _GROUP_KEY, _COUNTER_KEY)
pivot

fbi_code_desc,Aggravated Assault,Aggravated Battery,Arson,Burglary,Criminal Sexual Assault,Disorderly Conduct,Drug Abuse Violations,Embezzlement,Forgery and Counterfeiting,Fraud,...,Motor Vehicle Theft,Offenses Against Family and Children,Prostitution,Robbery,Sex Offense – Criminal Sexual Abuse,Simple Assault,Simple Battery,"Stolen Property (Buy, Receive, Possess)",Vandalism,Weapons Violations
year_month,,,,,,,,,,,,,,,,,,,,,
2001-01-01,546.0,967.0,67.0,1934.0,236.0,183.0,4675.0,16.0,149.0,1194.0,...,2097.0,52.0,563.0,1396.0,334.0,2600.0,5611.0,39.0,3966.0,337.0
2001-02-01,536.0,976.0,57.0,1666.0,163.0,195.0,4310.0,13.0,120.0,999.0,...,1785.0,37.0,426.0,1159.0,228.0,2322.0,5107.0,31.0,3665.0,301.0
2001-03-01,655.0,1184.0,93.0,1832.0,187.0,283.0,4822.0,7.0,168.0,1103.0,...,2152.0,53.0,550.0,1399.0,248.0,3172.0,6542.0,36.0,4618.0,345.0
2001-04-01,695.0,1502.0,89.0,1932.0,161.0,258.0,3999.0,8.0,177.0,981.0,...,2121.0,66.0,564.0,1341.0,243.0,2950.0,6873.0,34.0,4922.0,321.0
2001-05-01,688.0,1562.0,94.0,1997.0,190.0,267.0,4006.0,8.0,227.0,998.0,...,2197.0,61.0,503.0,1491.0,289.0,3083.0,7387.0,28.0,4758.0,391.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08-01,575.0,715.0,35.0,590.0,179.0,180.0,603.0,0.0,65.0,1154.0,...,1554.0,79.0,33.0,570.0,102.0,1697.0,3248.0,7.0,2363.0,470.0
2025-09-01,566.0,627.0,21.0,511.0,134.0,167.0,550.0,5.0,83.0,1083.0,...,1374.0,40.0,15.0,446.0,126.0,1705.0,3198.0,2.0,2293.0,503.0
2025-10-01,540.0,637.0,25.0,579.0,158.0,159.0,636.0,1.0,56.0,1069.0,...,1718.0,44.0,27.0,434.0,133.0,1646.0,3037.0,3.0,2427.0,418.0


In [10]:
# Remove scientific notation
# np.set_printoptions(suppress=True, precision=4, linewidth=100)

In [12]:
eps_grid = build_eps_grid(pivot)
print(eps_grid)

[1.00000000e-05 1.00000000e-04 1.00000000e-03 1.00000000e-02
 3.00000000e-02 5.00000000e-02 7.00000000e-02 1.00000000e-01
 1.58489319e-01 2.50000000e-01 3.92000000e-01 5.00000000e-01
 6.30957344e-01 1.00000000e+00]


In [10]:
def multiplicative_replacement(data, delta):
    """
    Equation (6) from Martín-Fernández et al. (2003)
    data : (T, D) array of counts
    delta: replacement value for zeros
    """
    data = data.astype(float)
    c = data.sum(axis=1, keepdims=True)          # row totals
    props = data / c                              # proportions
    
    Z_mask = (props == 0)                         # zero positions
    sum_delta = Z_mask.sum(axis=1, keepdims=True) * delta  # Σ δ_k per row
    
    result = props.copy()
    result[Z_mask] = delta
    result[~Z_mask] = (1 - sum_delta / 1.0)[~Z_mask.any(axis=1).reshape(-1,1) * np.ones_like(props, dtype=bool)] * props[~Z_mask]
    
    return result

[Sandford, R.F., Pierson, C.T., and Crovelli, R.A. (1993). An objective replacement method for censored geochemical data. Mathematical Geology, 25(1), 59–80](https://link.springer.com/article/10.1007/BF00890676)

In [ ]:
# constant
delta = 0.55

# Find the smallest nonzero value columns
zero_cols_idx = np.where((data == 0).any(axis=0))[0]

print(f"\n{' '*25} Zero Column Diagnostics:")

header = f"{'Idx':>5} | {'Min Nonzero':>12} | {'Geom Mean':>12} | {'Delta (0.65x)':>14} | {'Column Name'}"
print(header)
print("-" * len(header))

for j in zero_cols_idx:
    nonzero_vals = data[:, j][data[:, j] > 0]
    
    if len(nonzero_vals) == 0:
        continue  # safety guard
    
    min_nonzero  = nonzero_vals.min()
    geom_mean    = np.exp(np.log(nonzero_vals).mean())
    delta_j      = min_nonzero * delta
    col_name     = pivot.columns[j]

    print(f"{j:5d} | {min_nonzero:12.2f} | {geom_mean:12.2f} | {delta_j:14.4f} | {col_name}")

**Note:**
- **Min Nonzero** - the smallest count ever recorded in that column, ignoring zero months. For all 5 columns, this is 1, meaning the rarest observation is a single incident in a month.
- **Geom Mean** - the geometric mean of all nonzero months in that column. This is the "typical" monthly count at the time the crime occurred.
- **Delta** - the replacement value for zero months, set at 0.55 × min nonzero = 0.55 × 1 = 0.55 for all 5 columns.

### Zero Pattern Means
---
| Metric | Value | Interpretation |
|---|---|---|
| % of cells | 3.53% | **Rare zeros** -low contamination |
| Months with zeros | 252 / 300 | Zeros are **scattered across time** - not concentrated in one period |
| Columns with zeros | 5 / 26 | Only **5 categories** ever have zeros - the other 21 are clean |

**Key insight:** 5 specific categories occasionally hit zero across 252 months - this is a **structural sparsity** pattern, not random. Those 5 columns likely represent rare events or low-count categories.

In [ ]:
diag, clr_versions = sweep_eps_grid(pivot, eps_grid=cfg.config['_EPS_GRID'])
print(diag.to_string())
# Inspect CLR for chosen eps
# chosen_eps = 0.5
# chosen_clr = clr_versions[chosen_eps]

In [ ]:
# Find the smallest non-zero count in your entire pivot
min_nonzero = pivot[pivot > 0].min().min()

# ε = half the minimum non-zero observation
eps_data = min_nonzero / 2

print(f"Min non-zero count : {min_nonzero}")
print(f"Data-driven ε      : {eps_data}")

In [ ]:
import numpy as np

# 1. Pivot to (T × K) count matrix
# One row per month, one column per crime type
pivot = (
    fill_results['filled_df']
    .pivot(
        index=cfg.config["_DATE_KEY"],
        columns=cfg.config["_GROUP_KEY"],
        values="crime_count"
    )
    .sort_index()
)

# Pre-transform validation
# Enforce integer/float dtype - object columns break all numeric asserts silently
assert pivot.select_dtypes(exclude=[np.number]).empty, \
    "Pivot contains non-numeric columns - check fill_results dtype upstream."

assert not pivot.isnull().any().any(), \
    "Pivot contains NaN - missing month/crime combinations not filled."

assert (pivot >= 0).all().all(), \
    "Pivot contains negative counts - check fill_results upstream."

dead_cols = pivot.columns[pivot.sum(axis=0) == 0].tolist()
if dead_cols:
    raise ValueError(
        f"These crime types have zero counts across ALL months: {dead_cols}"
    )

# 2. Laplace smoothing
# ε = 0.5 is the Jeffreys prior pseudocount - standard for compositional data.
# Applied uniformly (not just to zeros) to preserve the ratio structure.
# Ref: Aitchison (1986), Martín-Fernández et al. (2003)
eps = 0.5
pivot_smoothed = pivot.add(eps)                                   # y_tk + ε

# 3. Monthly proportions
# π_t = (y_t + ε) / Σ_k(y_t + ε)
props_smoothed = pivot_smoothed.div(pivot_smoothed.sum(axis=1), axis=0)

assert np.allclose(props_smoothed.sum(axis=1), 1.0, atol=1e-10), \
    "Proportions do not sum to 1.0 per row - check smoothing step."

# 4. CLR transform
# clr_tk = log(π_tk) - (1/K) Σ_k log(π_tk)
log_props = np.log(props_smoothed)
clr       = log_props.sub(log_props.mean(axis=1), axis=0)

# Post-transform validation
assert clr.shape[0] == pivot.shape[0], "Row count mismatch after CLR."
assert clr.shape[1] == pivot.shape[1], "Column count mismatch after CLR."
assert np.allclose(clr.sum(axis=1), 0, atol=1e-10), \
    "CLR rows must sum to zero - rank deficiency check failed."

# Provenance metadata
clr.attrs["eps"]        = eps
clr.attrs["source"]     = "fill_results['filled_df']"
clr.attrs["date_key"]   = cfg.config["_DATE_KEY"]
clr.attrs["group_key"]  = cfg.config["_GROUP_KEY"]
clr.attrs["zero_cells"] = int((pivot == 0).sum().sum())

# Diagnostics
print(
    f"CLR matrix shape         : {clr.shape}\n"
    f"Date range               : {clr.index[0].strftime("%Y-%m")} -> {clr.index[-1].strftime("%Y-%m")}\n"
    f"Crime types (K)          : {clr.shape[1]}\n"
    f"Months (T)               : {clr.shape[0]}\n"
    f"Zero cells (pre-smooth)  : {clr.attrs['zero_cells']}\n"
    f"Pseudocount (ε)          : {eps}"
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# --- Sanity checks

# 1. Shape
print("Pivot shape :", pivot.shape)                 # (300, 26)
print("Proportions shape :", proportions.shape)     # (300, 26)
print("CLR shape :", clr.shape)                     # (300, 26)

# 2. Proportions sum to 1 per row (should be all ~1.0)
row_sums = proportions.sum(axis=1)
print(f"\nProportion row sums - min/max : {row_sums.min().round(6)} / {row_sums.max().round(6)}")

# 3. CLR rows sum to ~0 (fundamental CLR constraint)
clr_row_sums = clr.sum(axis=1)
print("CLR row sums - max abs :", clr_row_sums.abs().max().round(10))

# 4. Any remaining NaN or Inf after log?
print("CLR NaNs :", clr.isna().sum().sum())
print("CLR Infs :", np.isinf(clr.values).sum())

# 5. Zero-proportion check on raw (pre-pseudocount) pivot
zero_mask = (pivot == 0)
zero_pct_by_group = zero_mask.mean().sort_values(ascending=False)
print("\nZero % by crime type (top 8):")
print(zero_pct_by_group.head(8).round(4).to_string())
print()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 6. Preview CLR - representative crime types
# zero_pct_by_group.index[0]  = sparsest crime type (most zeros)
# zero_pct_by_group.index[-1] = densest crime type (fewest zeros)
cols_to_plot = [
    zero_pct_by_group.index[0],   # sparsest
    zero_pct_by_group.index[-1],  # densest
    "Homicide",                   # adjust name to match your labels
    "Theft",
]

cols_to_plot = [c for c in cols_to_plot if c in clr.columns]


if len(cols_to_plot) == 0:
    print("WARNING: No valid columns to plot - check cols_to_plot against clr.columns")
else:
    # Convert index once
    clr_index_str  = clr.index.strftime("%Y-%m")

    # COVID reference date
    covid_date     = "2020-03"
    covid_in_index = covid_date in clr_index_str.values

    # Post-COVID
    post_date      = "2023-01"
    post_in_index = post_date in clr_index_str.values

    # Build figure - sharex=False so each subplot manages its own x-axis
    fig, axes = plt.subplots(
        len(cols_to_plot), 1,
        figsize=(14, 2.5 * len(cols_to_plot)),
        sharex=False
    )
    axes = np.atleast_1d(axes)

    fig.suptitle(
        "CLR Time Series - Chicago Crime Composition",
        fontsize=12, fontweight="bold", y=1.01
    )

    # X-axis ticks - derived from data length, not hardcoded
    n_ticks  = min(50, len(clr))
    tick_idx = np.linspace(0, len(clr) - 1, n_ticks, dtype=int)

    for ax, col in zip(axes, cols_to_plot):
        ax.plot(clr_index_str, clr[col], lw=0.9, color="steelblue")
        ax.axhline(0, color="gray", lw=0.8, ls="--")
        ax.set_title(f"CLR - {col}", fontsize=10)

        if covid_in_index:
            ax.axvline(
                covid_date,
                color="red", lw=1.0, ls=":",
                label="COVID onset (Mar 2020)"
            )

        if post_in_index:
                ax.axvline(
                post_date,
                color="green", lw=1.0, ls=":",
                label="Post-COVID (Jan 2023)"
            )
            
        ax.legend(fontsize=8, loc="upper left")

        # Apply ticks to every subplot
        ax.set_xticks(tick_idx)
        ax.set_xticklabels(
            clr_index_str[tick_idx],
            rotation=45, ha="right", fontsize=8
        )

    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 6. Plot CLR - all crime types
cols_to_plot = [c for c in clr.columns if c in clr.columns]

if len(cols_to_plot) == 0:
    print("WARNING: No valid columns to plot - check clr.columns")
else:
    # Convert index once
    clr_index_str = clr.index.strftime("%Y-%m-%d")

    # Reference line positions
    covid_date     = "2020-03-01"
    postcovid_date = "2023-01-01"
    covid_pos      = int(np.searchsorted(clr_index_str.values, covid_date))
    postcovid_pos  = int(np.searchsorted(clr_index_str.values, postcovid_date))

    # Grid layout
    n_cols   = 4
    n_rows   = int(np.ceil(len(cols_to_plot) / n_cols))
    n_ticks  = min(6, len(clr))
    tick_idx = np.linspace(0, len(clr) - 1, n_ticks, dtype=int)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(22, 3.5 * n_rows),
        sharex=False
    )
    axes = axes.flatten()

    fig.suptitle(
        "CLR Time Series - All Chicago Crime Categories (2001–2025)",
        fontsize=13, fontweight="bold", y=1.01
    )

    for i, col in enumerate(cols_to_plot):
        ax = axes[i]

        ax.plot(range(len(clr)), clr[col], lw=0.8, color="steelblue")
        ax.axhline(0, color="gray", lw=0.6, ls="--")

        # COVID onset - red
        ax.axvline(
            covid_pos,
            color="red", lw=0.8, ls=":",
            label="COVID (Mar 2020)"
        )

        # Post-COVID - green
        ax.axvline(
            postcovid_pos,
            color="green", lw=0.8, ls=":",
            label="Post-COVID (Jan 2023)"
        )

        ax.set_title(col, fontsize=7, pad=3)
        ax.tick_params(axis='y', labelsize=6)
        ax.set_xticks(tick_idx)
        ax.set_xticklabels(
            clr_index_str.values[tick_idx],
            rotation=45, ha="right", fontsize=5
        )

        # Legend only on first subplot
        if i == 0:
            ax.legend(fontsize=6, loc="upper left")

    # Hide unused subplots
    for j in range(len(cols_to_plot), len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()